# Hexagonal-only geometric sampling by trajectory region

Input: the endpoint-cleaned dataset from `pre_processing4.ipynb`.
This version follows `preprocessing_hexagonals.ipynb` and changes only the sampling scope:

- `hexagonal`: apply the original region-aware geometric selection and minimum quotas.
- `needle` and `dist_perovskite`: keep every available frame; bypass selection entirely.
- All `static` configurations: keep unchanged.

The original notebook remains in `outputs/new_freeze3`. This run writes to `outputs/only_hex_new`.
The new dataset is `configurations_only_hex_new.extxyz`. Records are copied byte-for-byte from
the cleaned input. Sampling decisions and region annotations are stored in separate CSV files.
No energy, force, or stress label is used to select frames. Endpoint cleaning is not repeated.
The next notebook reuses the exact OLD formula assignment for the 64/8/8 split.

In [1]:
import copy
import hashlib
import json
import math
import os
import platform
from collections import defaultdict
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mplconfig-hexagonal-stratified')
import ase
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from ase.geometry import find_mic
from ase.io import read, write
from IPython.display import display

def find_project_dir():
    if os.environ.get('ABS3_PROJECT_DIR'):
        return Path(os.environ['ABS3_PROJECT_DIR']).expanduser().resolve()
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / 'data/stratification_split.ipynb').is_file():
            return root
    raise FileNotFoundError('Set ABS3_PROJECT_DIR to the project root.')

PROJECT_DIR = find_project_dir()
NOTEBOOK_DIR = PROJECT_DIR / 'data/Preprocess notebooks'
SOURCE_EXTXYZ = Path(os.environ.get('ABS3_HEX_SOURCE', str(
    NOTEBOOK_DIR / 'outputs/old_gradual/configurations_results1_no_meta_endpoint_cleaned/extxyz/configurations_endpoint_cleaned.extxyz'
))).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get('ABS3_HEX_OUTPUT', str(
    NOTEBOOK_DIR / 'outputs/only_hex_new'
))).expanduser().resolve()
REPORT_DIR = OUTPUT_DIR / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
WRITE_OUTPUTS = True
assert not (OUTPUT_DIR / 'manifest.json').exists(), 'Existing run: choose a new ABS3_HEX_OUTPUT directory'
assert not (OUTPUT_DIR / 'configurations_only_hex_new.extxyz').exists(), 'Dataset already exists'
EARLY_END, MIDDLE_END = 0.2, 0.8
REGIONS = ['early', 'middle', 'late', 'static']
POLYMORPHS = ['dist_perovskite', 'hexagonal', 'needle']
REGION_COLORS = dict(zip(REGIONS, ['#4477AA', '#EEAA33', '#CC6677', '#228833']))

# Preserve the historical parameters; do not tune them using labels or test results.
# Base geometry tolerances come from C5 in pre_processing5.
BASE_RMS_A = 0.075
BASE_CELL_REL = 0.0075
POLICIES = {
    (False, False): dict(min_count=3, min_fraction=0.10, density_scale=1.0),
    (True,  False): dict(min_count=5, min_fraction=0.20, density_scale=0.75),
    (False, True):  dict(min_count=5, min_fraction=0.25, density_scale=0.75),
    (True,  True):  dict(min_count=8, min_fraction=0.40, density_scale=0.5625),
}
EXAMPLE_TRAJECTORY = 'Mg1Nb1S3--Mg1Nb1S3_hexagonal--fm--gga-relax'
# Export one page per trajectory, including those not displayed inline.
EXPORT_ALL_TRAJECTORIES_PDF = True

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

assert SOURCE_EXTXYZ.is_file(), SOURCE_EXTXYZ
assert 0 < EARLY_END < MIDDLE_END < 1
assert BASE_RMS_A > 0 and BASE_CELL_REL > 0
for policy in POLICIES.values():
    assert policy['min_count'] >= 1 and 0 < policy['min_fraction'] <= 1
    assert 0 < policy['density_scale'] <= 1
if WRITE_OUTPUTS:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 160,
                     'axes.grid': True, 'grid.alpha': 0.2,
                     'axes.spines.top': False, 'axes.spines.right': False})
source_hash = sha256_file(SOURCE_EXTXYZ)
print('Input:', SOURCE_EXTXYZ)
print('Output:', OUTPUT_DIR)
display(pd.DataFrame([dict(hexagonal=h, late=l, **p) for (h,l),p in POLICIES.items()]))

Input: /users/0/cmadridl/projects/dmc_cristelle-madrid-lopez/projects/ABS3-fine-tuning/dev/data/cml/data/Preprocess notebooks/outputs/old_gradual/configurations_results1_no_meta_endpoint_cleaned/extxyz/configurations_endpoint_cleaned.extxyz
Output: /users/0/cmadridl/projects/dmc_cristelle-madrid-lopez/projects/ABS3-fine-tuning/dev/data/cml/data/Preprocess notebooks/outputs/only_hex_new


,hexagonal,late,min_count,min_fraction,density_scale
0,False,False,3,0.10,1.0000
1,True,False,5,0.20,0.7500
2,False,True,5,0.25,0.7500
3,True,True,8,0.40,0.5625


## Input integrity and trajectory regions

Preserve original indices. Progress is the ordinal rank among available endpoint-cleaned
relaxation frames divided by `max(n-1, 1)`. A one-frame trajectory belongs to `early`.
Regions are `early` (progress <= 0.2), `middle` (0.2 < progress <= 0.8), and `late` (> 0.8).
Static configurations are paired by replacing `--gga-relax` with `--gga-static`.
No removed endpoint is restored. Regions are annotations only for the non-target polymorphs.

In [2]:
atoms_all = read(SOURCE_EXTXYZ, index=':', format='extxyz')
assert atoms_all, 'Empty input'
required = {'config_id', 'source_key', 'formula', 'polymorph', 'data_origin', 'frame_index'}
trajectories, statics = defaultdict(list), {}
rows = []
for source_index, atoms in enumerate(atoms_all):
    assert not required.difference(atoms.info), (source_index, atoms.info)
    info = atoms.info
    assert info['data_origin'] in {'relax_trajectory', 'static'}
    assert info['polymorph'] in POLYMORPHS
    assert atoms.get_volume() > 0 and len(atoms) > 0
    assert atoms.get_forces().shape == (len(atoms), 3)
    assert atoms.get_stress(voigt=False).shape == (3, 3)
    for value in (atoms.positions, atoms.cell.array, atoms.get_potential_energy(),
                  atoms.get_forces(), atoms.get_stress(voigt=False)):
        assert np.isfinite(value).all(), (source_index, info['config_id'])
    if info['data_origin'] == 'static':
        assert info['source_key'] not in statics, 'Duplicate static'
        statics[info['source_key']] = source_index
    else:
        trajectories[info['source_key']].append(source_index)
    rows.append(dict(source_index=source_index, **{k: info[k] for k in sorted(required)},
                     energy_eV_atom=atoms.get_potential_energy()/len(atoms),
                     fmax_eV_A=float(np.linalg.norm(atoms.get_forces(), axis=1).max())))
inventory = pd.DataFrame(rows).set_index('source_index', drop=False)
assert inventory.config_id.is_unique
inventory['trajectory_region'] = 'static'
inventory['trajectory_progress'] = np.nan
inventory['trajectory_key'] = ''
paired_static = {}
for key in sorted(trajectories):
    ids = sorted(trajectories[key], key=lambda i: int(atoms_all[i].info['frame_index']))
    trajectories[key] = ids
    frame_indices = inventory.loc[ids, 'frame_index'].to_numpy(dtype=int)
    assert np.all(np.diff(frame_indices) > 0), key
    assert key.endswith('--gga-relax'), key
    static_key = key[:-len('--gga-relax')] + '--gga-static'
    assert static_key in statics, key
    static_id = statics[static_key]
    paired_static[key] = static_id
    reference = atoms_all[ids[0]]
    for i in ids + [static_id]:
        a = atoms_all[i]
        np.testing.assert_array_equal(a.numbers, reference.numbers)
        np.testing.assert_array_equal(a.pbc, reference.pbc)
        assert a.info['formula'] == reference.info['formula']
        assert a.info['polymorph'] == reference.info['polymorph']
    progress = np.arange(len(ids)) / max(len(ids)-1, 1)
    regions = np.where(progress <= EARLY_END, 'early',
                       np.where(progress <= MIDDLE_END, 'middle', 'late'))
    inventory.loc[ids, 'trajectory_progress'] = progress
    inventory.loc[ids, 'trajectory_region'] = regions
    inventory.loc[ids + [static_id], 'trajectory_key'] = key
assert set(paired_static.values()) == set(statics.values()), 'Static without a relaxation pair'
assert inventory.trajectory_key.ne('').all()
print(f'{len(atoms_all):,} configurations; {len(trajectories)} trajectories; {len(statics)} static')
display(pd.crosstab(inventory.polymorph, inventory.trajectory_region).reindex(columns=REGIONS))

13,874 configurations; 240 trajectories; 240 static


trajectory_region,early,middle,late,static
polymorph,,,,
dist_perovskite,590,1635,576,80
hexagonal,1634,4772,1623,80
needle,593,1637,574,80


## Original geometric selection, applied only to hexagonal trajectories

Compare atoms in their original order using fractional displacements in the midpoint cell.
`ase.geometry.find_mic` handles periodic images. Relative cell change uses the symmetric
Frobenius norm. The distance is `max(RMS_MIC / tolerance_R, relative_cell_change / tolerance_H)`.

Keep a frame when its distance to every previously retained frame in its trajectory region
is at least one. Keep the first point in each region. Then add the farthest remaining points
until the minimum quota is met; temporal order breaks ties.
The quota is `min(n, max(min_count, ceil(min_fraction * n)))`.

Hexagonal early/middle use `(5, 0.20, 0.75)` and late uses `(8, 0.40, 0.5625)` for
`(min_count, min_fraction, density_scale)`. Base tolerances remain 0.075 A and 0.0075.
The historical non-hexagonal policies remain documented but are never called.
Non-target frames receive `keep_not_targeted` without a geometric selection score.

In [3]:
# Geometry cache; selection does not use DFT labels.
cells_array = [a.cell.array.copy() for a in atoms_all]
fractional = [a.get_scaled_positions(wrap=False) for a in atoms_all]
geometry_cache = {}

def geometric_metrics(i, j):
    if i == j:
        return (0.0, 0.0)
    key = (min(i,j), max(i,j))
    if key not in geometry_cache:
        a, b = key
        if not np.array_equal(atoms_all[a].numbers, atoms_all[b].numbers):
            raise ValueError('Incompatible species order')
        midpoint = 0.5 * (cells_array[a] + cells_array[b])
        displacement = (fractional[b] - fractional[a]) @ midpoint
        _, lengths = find_mic(displacement, cell=midpoint, pbc=atoms_all[a].pbc)
        rms = float(np.sqrt(np.mean(lengths**2)))
        cell_rel = float(np.linalg.norm(cells_array[b]-cells_array[a]) / np.linalg.norm(midpoint))
        assert np.isfinite([rms, cell_rel]).all()
        geometry_cache[key] = (rms, cell_rel)
    return geometry_cache[key]

def score(i, j, scale=1.0):
    rms, cell_rel = geometric_metrics(i,j)
    return max(rms/(BASE_RMS_A*scale), cell_rel/(BASE_CELL_REL*scale))

def select_stratum(ids, min_count, min_fraction, density_scale):
    # IDs are already in temporal order; preserve the input.
    quota = min(len(ids), max(min_count, math.ceil(min_fraction*len(ids))))
    selected, decisions = [], []
    for i in ids:
        if not selected:
            record = dict(source_index=i, novelty_score=np.nan, nearest_score=np.nan,
                          reference_source_index=-1, reference_config_id='',
                          delta_r_rms_A=np.nan, cell_change_relative=np.nan,
                          geometric_keep=True, keep=True, keep_reason='region_anchor')
        else:
            ref = selected[-1]
            distances = [score(i, j, density_scale) for j in selected]
            nearest = min(distances)
            rms, cell_rel = geometric_metrics(ref, i)
            keep = nearest >= 1.0
            record = dict(source_index=i, novelty_score=distances[-1], nearest_score=nearest,
                          reference_source_index=ref, reference_config_id=atoms_all[ref].info['config_id'],
                          delta_r_rms_A=rms, cell_change_relative=cell_rel,
                          geometric_keep=keep, keep=keep,
                          keep_reason='geometric_novelty' if keep else 'geometric_redundancy')
        if record['keep']:
            selected.append(i)
        record.update(required_minimum=quota, density_scale=density_scale,
                      rms_tolerance_A=BASE_RMS_A*density_scale,
                      cell_tolerance_relative=BASE_CELL_REL*density_scale,
                      quota_selection_score=np.nan)
        decisions.append(record)
    nearest_remaining = {
        i: min(score(i,j,density_scale) for j in selected)
        for i in ids if i not in selected
    } if len(selected) < quota else {}
    lookup = {r['source_index']: r for r in decisions}
    while len(selected) < quota:
        chosen = max(nearest_remaining, key=nearest_remaining.get)
        lookup[chosen].update(keep=True, keep_reason='minimum_quota',
                              quota_selection_score=nearest_remaining.pop(chosen))
        selected.append(chosen)
        for i in nearest_remaining:
            nearest_remaining[i] = min(nearest_remaining[i], score(i,chosen,density_scale))
    assert len(selected) >= quota
    return decisions

decision_rows = []
for number, key in enumerate(sorted(trajectories), 1):
    ids = trajectories[key]
    is_hex = atoms_all[ids[0]].info['polymorph'] == 'hexagonal'
    for region in REGIONS[:-1]:
        region_ids = [i for i in ids if inventory.at[i, 'trajectory_region'] == region]
        if region_ids and is_hex:
            decision_rows.extend(select_stratum(region_ids, **POLICIES[(True, region=='late')]))
        elif region_ids:
            # Non-target polymorphs bypass geometric sampling and quotas completely.
            decision_rows.extend(dict(
                source_index=i, keep=True, geometric_keep=False,
                keep_reason='keep_not_targeted', required_minimum=len(region_ids),
                reference_source_index=-1, reference_config_id='',
                novelty_score=np.nan, nearest_score=np.nan,
                delta_r_rms_A=np.nan, cell_change_relative=np.nan,
                density_scale=np.nan, rms_tolerance_A=np.nan,
                cell_tolerance_relative=np.nan, quota_selection_score=np.nan,
            ) for i in region_ids)
    static_id = paired_static[key]
    decision_rows.append(dict(source_index=static_id, keep=True, geometric_keep=True,
                             keep_reason='protected_static', required_minimum=1,
                             reference_source_index=-1, reference_config_id='',
                             density_scale=1.0, rms_tolerance_A=BASE_RMS_A,
                             cell_tolerance_relative=BASE_CELL_REL))
    if number % 40 == 0:
        print(f'Processed {number}/{len(trajectories)} trajectories', flush=True)
decisions = inventory.reset_index(drop=True).merge(pd.DataFrame(decision_rows), on='source_index', validate='one_to_one')
decisions = decisions.sort_values('source_index').reset_index(drop=True)
assert len(decisions) == len(atoms_all)
assert decisions.source_index.tolist() == list(range(len(atoms_all)))
selected_ids = set(decisions.loc[decisions.keep, 'source_index'])
print('Retained:', len(selected_ids), 'of', len(atoms_all))
display(pd.crosstab([decisions.polymorph, decisions.trajectory_region], decisions.keep))

Processed 40/240 trajectories


Processed 80/240 trajectories


Processed 120/240 trajectories


Processed 160/240 trajectories


Processed 200/240 trajectories


Processed 240/240 trajectories


Retained: 8898 of 13874


keep                               False  True 
polymorph       trajectory_region              
dist_perovskite early                  0    590
                late                   0    576
                middle                 0   1635
                static                 0     80
hexagonal       early                862    772
                late                 901    722
                middle              3213   1559
                static                 0     80
needle          early                  0    593
                late                   0    574
                middle                 0   1637
                static                 0     80

## Composition and distances to static configurations

Report retention by trajectory, region, and polymorph. Static-to-frame distances use the base
tolerances for every polymorph and serve only as diagnostics. They do not affect selection.
The summaries include the closest point in each region and the last original/retained frame.

In [4]:
stratum_summary = decisions.groupby(['trajectory_key', 'formula', 'polymorph', 'trajectory_region'], sort=True).agg(
    frames_before=('keep','size'), frames_kept=('keep','sum'),
    required_minimum=('required_minimum','first'), density_scale=('density_scale','first'),
    quota_added=('keep_reason', lambda s: int(s.eq('minimum_quota').sum())),
).reset_index()
stratum_summary['frames_dropped'] = stratum_summary.frames_before-stratum_summary.frames_kept
stratum_summary['retention_fraction'] = stratum_summary.frames_kept/stratum_summary.frames_before
assert stratum_summary.frames_kept.ge(stratum_summary.required_minimum).all()

def max_drop_run(keep):
    longest = current = 0
    for value in keep:
        current = 0 if value else current + 1
        longest = max(longest, current)
    return longest

trajectory_rows, static_frame_rows, static_summary_rows = [], [], []
decision_lookup = decisions.set_index('source_index')
for key in sorted(trajectories):
    ids = trajectories[key]
    static_id = paired_static[key]
    kept = [i for i in ids if i in selected_ids]
    indices = [int(inventory.at[i,'frame_index']) for i in kept]
    row = dict(trajectory_key=key, formula=inventory.at[ids[0],'formula'],
               polymorph=inventory.at[ids[0],'polymorph'], frames_before=len(ids),
               frames_kept=len(kept), frames_dropped=len(ids)-len(kept),
               retention_fraction=len(kept)/len(ids),
               max_kept_index_gap=max(np.diff(indices), default=0),
               max_consecutive_drops=max_drop_run([i in selected_ids for i in ids]),
               static_kept=static_id in selected_ids,
               last_original_frame_kept=ids[-1] in selected_ids)
    for region in REGIONS[:-1]:
        region_ids = [i for i in ids if inventory.at[i,'trajectory_region'] == region]
        row[region+'_before'] = len(region_ids)
        row[region+'_kept'] = sum(i in selected_ids for i in region_ids)
    for i in ids:
        rms, cell_rel = geometric_metrics(static_id, i)
        static_frame_rows.append(dict(trajectory_key=key, static_config_id=inventory.at[static_id,'config_id'],
            config_id=inventory.at[i,'config_id'], source_index=i,
            frame_index=int(inventory.at[i,'frame_index']), trajectory_region=inventory.at[i,'trajectory_region'],
            keep=i in selected_ids, rms_to_static_A=rms, cell_to_static_relative=cell_rel,
            distance_to_static_base=score(static_id,i)))
    for name, i in [('last_original', ids[-1]), ('last_kept', kept[-1])]:
        rms, cell_rel = geometric_metrics(static_id,i)
        row['static_to_'+name+'_rms_A'] = rms
        row['static_to_'+name+'_cell_relative'] = cell_rel
        row['static_to_'+name+'_base_score'] = score(static_id,i)
    trajectory_rows.append(row)

trajectory_summary = pd.DataFrame(trajectory_rows)
static_frame_distances = pd.DataFrame(static_frame_rows)
for (key, region), group in static_frame_distances.groupby(['trajectory_key','trajectory_region']):
    closest = group.loc[group.distance_to_static_base.idxmin()]
    closest_kept = group.loc[group.keep].sort_values(['distance_to_static_base','frame_index']).iloc[0]
    static_summary_rows.append(dict(trajectory_key=key, trajectory_region=region,
        closest_config_id=closest.config_id, closest_frame_index=int(closest.frame_index),
        min_base_score=float(closest.distance_to_static_base),
        median_base_score=float(group.distance_to_static_base.median()),
        max_base_score=float(group.distance_to_static_base.max()),
        closest_rms_A=float(closest.rms_to_static_A),
        closest_cell_relative=float(closest.cell_to_static_relative),
        min_retained_base_score=float(closest_kept.distance_to_static_base),
        closest_retained_config_id=closest_kept.config_id))
static_region_summary = pd.DataFrame(static_summary_rows)
composition = decisions.groupby(['polymorph','trajectory_region']).agg(
    before=('keep','size'), after=('keep','sum')).reindex(
        pd.MultiIndex.from_product([POLYMORPHS,REGIONS], names=['polymorph','trajectory_region']), fill_value=0).reset_index()
composition['retention_fraction'] = composition.after/composition.before.replace(0,np.nan)
display(composition)
display(trajectory_summary.sort_values('max_consecutive_drops', ascending=False).head(15))
display(static_region_summary.head(12))

,polymorph,trajectory_region,before,after,retention_fraction
0,dist_perovskite,early,590,590,1.000000
1,dist_perovskite,middle,1635,1635,1.000000
2,dist_perovskite,late,576,576,1.000000
3,dist_perovskite,static,80,80,1.000000
4,hexagonal,early,1634,772,0.472460
5,hexagonal,middle,4772,1559,0.326697
6,hexagonal,late,1623,722,0.444855
7,hexagonal,static,80,80,1.000000
8,needle,early,593,593,1.000000
9,needle,middle,1637,1637,1.000000


,trajectory_key,formula,polymorph,frames_before,frames_kept,frames_dropped,retention_fraction,max_kept_index_gap,max_consecutive_drops,static_kept,...,middle_before,middle_kept,late_before,late_kept,static_to_last_original_rms_A,static_to_last_original_cell_relative,static_to_last_original_base_score,static_to_last_kept_rms_A,static_to_last_kept_cell_relative,static_to_last_kept_base_score
46,Ca1S3Ti1--Ca1S3Ti1_hexagonal--fm--gga-relax,Ca1S3Ti1,hexagonal,143,38,105,0.265734,18,17,True,...,85,17,29,12,0.002227,0.000243,0.032391,0.002227,0.000243,0.032391
130,Hf1S3Sn1--Hf1S3Sn1_hexagonal--nm--gga-relax,Hf1S3Sn1,hexagonal,183,52,131,0.284153,18,17,True,...,109,22,37,15,0.002096,0.000163,0.027943,0.004160,0.000374,0.055461
73,Cd1S3Ti1--Cd1S3Ti1_hexagonal--fm--gga-relax,Cd1S3Ti1,hexagonal,149,45,104,0.302013,17,16,True,...,89,18,30,12,0.003895,0.000663,0.088419,0.003895,0.000663,0.088419
43,Ca1S3Sn1--Ca1S3Sn1_hexagonal--nm--gga-relax,Ca1S3Sn1,hexagonal,150,51,99,0.340000,17,16,True,...,90,18,30,12,0.002967,0.000563,0.075086,0.002967,0.000563,0.075086
226,S3Sr1V1--S3Sr1V1_hexagonal--fm--gga-relax,S3Sr1V1,hexagonal,155,39,116,0.251613,16,15,True,...,93,19,31,13,0.001349,0.000208,0.027700,0.001349,0.000208,0.027700
166,Nb1S3Sr1--Nb1S3Sr1_hexagonal--fm--gga-relax,Nb1S3Sr1,hexagonal,113,32,81,0.283186,16,15,True,...,67,17,23,10,0.001782,0.000249,0.033200,0.001782,0.000249,0.033200
139,Mg1Nb1S3--Mg1Nb1S3_hexagonal--fm--gga-relax,Mg1Nb1S3,hexagonal,196,72,124,0.367347,16,15,True,...,117,36,39,16,0.002012,0.000587,0.078257,0.006797,0.000964,0.128557
217,S3Sn1Zr1--S3Sn1Zr1_hexagonal--nm--gga-relax,S3Sn1Zr1,hexagonal,119,39,80,0.327731,16,15,True,...,71,19,24,10,0.004022,0.000481,0.064198,0.004022,0.000481,0.064198
202,S3Si1Zn1--S3Si1Zn1_hexagonal--nm--gga-relax,S3Si1Zn1,hexagonal,111,36,75,0.324324,15,14,True,...,66,17,22,9,0.002960,0.000346,0.046142,0.002960,0.000346,0.046142
115,Ge1S3Sn1--Ge1S3Sn1_hexagonal--nm--gga-relax,Ge1S3Sn1,hexagonal,177,70,107,0.395480,15,14,True,...,105,32,36,15,0.003160,0.000581,0.077445,0.001846,0.000451,0.060106


,trajectory_key,trajectory_region,closest_config_id,closest_frame_index,min_base_score,median_base_score,max_base_score,closest_rms_A,closest_cell_relative,min_retained_base_score,closest_retained_config_id
0,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-relax,early,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-re...,5,20.465143,20.754088,21.103962,0.584617,0.153489,20.465143,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-re...
1,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-relax,late,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-re...,117,0.024707,0.389724,0.896076,0.001626,0.000185,0.024707,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-re...
2,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-relax,middle,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-re...,91,0.900450,3.085426,20.537182,0.067534,0.004623,0.900450,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-re...
3,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax,early,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax::t...,23,25.488865,26.835219,27.148399,0.859050,0.191166,25.636729,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax::t...
4,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax,late,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax::t...,115,0.035899,0.346108,1.008369,0.002410,0.000269,0.035899,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax::t...
5,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax,middle,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax::t...,92,1.187383,8.521510,25.199551,0.062102,0.008905,1.430936,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax::t...
6,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax,early,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax::traj...,0,0.678991,0.773415,0.857962,0.050924,0.004816,0.678991,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax::traj...
7,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax,late,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax::traj...,11,0.066726,0.069609,0.098805,0.004531,0.000500,0.066726,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax::traj...
8,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax,middle,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax::traj...,9,0.139223,0.277088,0.846425,0.010442,0.000889,0.139223,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax::traj...
9,Ba1Hf1S3--Ba1Hf1S3_dist_perovskite--nm--gga-relax,early,Ba1Hf1S3--Ba1Hf1S3_dist_perovskite--nm--gga-re...,3,0.524646,0.678794,0.807861,0.039348,0.000880,0.524646,Ba1Hf1S3--Ba1Hf1S3_dist_perovskite--nm--gga-re...


## Visual inspection of retained configurations

Each page shows geometric novelty relative to the last retained point in the first pass,
energy per atom, and distance to static. Quota additions and region anchors have separate markers.
For non-target trajectories, the first panel explicitly states that all frames were retained.
Static is plotted separately with a star; its horizontal position is only a display convention.
The dashed line to static does not represent an additional relaxation step.

In [5]:
def save_show(fig, name):
    fig.tight_layout()
    if WRITE_OUTPUTS:
        fig.savefig(FIGURE_DIR/name, bbox_inches='tight')
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(1,2,figsize=(14,4.5))
for polymorph in POLYMORPHS:
    sub = trajectory_summary[trajectory_summary.polymorph == polymorph]
    axes[0].scatter(sub.frames_before, sub.frames_kept, label=polymorph, alpha=.7)
upper = trajectory_summary.frames_before.max()
axes[0].plot([0,upper],[0,upper],'k--',lw=1)
axes[0].set(xlabel='Relaxation frames before / trajectory',ylabel='Retained relaxation frames',title='Trajectory summary')
axes[0].legend()
composition.pivot(index='polymorph',columns='trajectory_region',values='retention_fraction').reindex(
    columns=REGIONS).plot.bar(ax=axes[1], color=[REGION_COLORS[r] for r in REGIONS])
axes[1].set(ylabel='Retained fraction',xlabel='',title='Retention by polymorph and region',ylim=(0,1.1))
axes[1].tick_params(axis='x',rotation=15)
save_show(fig,'01_trajectory_screening.png')

def plot_trajectory(key):
    ids = trajectories[key]
    static_id = paired_static[key]
    group = decision_lookup.loc[ids]
    x = group.frame_index.to_numpy(dtype=int)
    span = max(int(x[-1]-x[0]),1)
    static_x = float(x[-1] + max(2, .06*span))
    kept = [i for i in ids if i in selected_ids]
    fig, axes = plt.subplots(3,1,figsize=(13,10),sharex=True)
    fig.suptitle(key,fontsize=11)
    for region in REGIONS[:-1]:
        sub = group[group.trajectory_region == region]
        if sub.empty:
            continue
        left,right = sub.frame_index.min()-.5,sub.frame_index.max()+.5
        for ax in axes:
            ax.axvspan(left,right,color=REGION_COLORS[region],alpha=.12)
        axes[0].text((left+right)/2,.96,region,va='top',transform=axes[0].get_xaxis_transform(),
                     ha='center',color=REGION_COLORS[region])
        axes[0].plot(sub.frame_index,sub.novelty_score,'o-',color='.6',ms=3,lw=.8)
    geometric = group.keep_reason.eq('geometric_novelty')
    quota = group.keep_reason.eq('minimum_quota')
    anchors = group.keep_reason.eq('region_anchor')
    axes[0].scatter(group.loc[geometric,'frame_index'],group.loc[geometric,'novelty_score'],
                    color='#CC6677',label='KEEP geometry',zorder=4,s=23)
    axes[0].scatter(group.loc[quota,'frame_index'],group.loc[quota,'novelty_score'],
                    color='#AA7700',marker='D',label='KEEP quota',zorder=5,s=26)
    axes[0].scatter(group.loc[anchors,'frame_index'],np.ones(anchors.sum()),
                    color='#4477AA',marker='s',label='Anchor (no score)',s=25)
    static_score = score(static_id,kept[-1])
    axes[0].scatter([static_x],[static_score],marker='*',s=170,color=REGION_COLORS['static'],
                    label='static vs last KEEP (base)',zorder=6)
    axes[0].axhline(1,color='black',ls='--',label='Geometric threshold')
    axes[0].set_yscale('symlog',linthresh=.01)
    axes[0].set(ylabel='max(metric / tolerance)',
                title='Novelty: reference = last geometric KEEP in the region')
    axes[0].set_ylim(0, 1.3*max(1.0, static_score, group.novelty_score.max()))
    if group.keep_reason.eq('keep_not_targeted').all():
        axes[0].set_title('All non-target frames retained; geometric selection was bypassed')
    axes[0].legend(fontsize=8,ncol=3,loc='lower left')
    axes[1].plot(x,group.energy_eV_atom,color='.6',lw=1)
    for region in REGIONS[:-1]:
        mask = group.keep & group.trajectory_region.eq(region)
        axes[1].scatter(group.loc[mask,'frame_index'],group.loc[mask,'energy_eV_atom'],
                        color=REGION_COLORS[region],s=15,label=region)
    axes[1].scatter([static_x],[inventory.at[static_id,'energy_eV_atom']],marker='*',s=170,
                    color=REGION_COLORS['static'],label='static',zorder=6)
    axes[1].set(ylabel='E [eV/atom]',title='Energy diagnostic (not used for selection)')
    axes[1].legend(ncol=4,fontsize=8)
    distances = static_frame_distances[static_frame_distances.trajectory_key==key]
    for region in REGIONS[:-1]:
        sub = distances[distances.trajectory_region==region]
        axes[2].plot(sub.frame_index,sub.distance_to_static_base,'o-',ms=3,lw=.8,
                     color=REGION_COLORS[region],label=region)
    axes[2].scatter([static_x],[0],marker='*',s=170,color=REGION_COLORS['static'],zorder=6,label='static (0)')
    axes[2].plot([x[-1],static_x],[distances.iloc[-1].distance_to_static_base,0],
                 '--',color=REGION_COLORS['static'],lw=1)
    axes[2].set_yscale('symlog',linthresh=.01)
    axes[2].set_ylim(bottom=0)
    axes[2].axhline(1,color='black',ls='--',lw=1)
    axes[2].set(ylabel='Distance to static / base tolerance',xlabel='Original frame_index; static shown separately',
                title='Geometric distance from static to all regions')
    ticks = [float(v) for v in np.linspace(x[0],x[-1],min(6,len(x)))]
    axes[2].set_xticks(ticks+[static_x], [str(int(round(v))) for v in ticks]+['static'])
    for ax in axes:
        ax.axvline((x[-1]+static_x)/2,color=REGION_COLORS['static'],ls=':',lw=1)
    fig.tight_layout(rect=(0,0,1,.97))
    return fig

example_key = EXAMPLE_TRAJECTORY if EXAMPLE_TRAJECTORY in trajectories else trajectory_summary.sort_values(
    ['frames_before','trajectory_key'],ascending=[False,True]).iloc[0].trajectory_key
save_show(plot_trajectory(example_key),'03_accumulated_novelty_example_with_static.png')

# Apply the same diagnostics to all trajectories, including non-target polymorphs.
if WRITE_OUTPUTS and EXPORT_ALL_TRAJECTORIES_PDF:
    with PdfPages(FIGURE_DIR/'trajectory_screening_all.pdf') as pdf:
        for number,key in enumerate(sorted(trajectories),1):
            fig = plot_trajectory(key)
            pdf.savefig(fig)
            plt.close(fig)
            if number % 40 == 0:
                print(f'PDF: {number}/{len(trajectories)} trajectories',flush=True)

fig,ax = plt.subplots(figsize=(9,4.5))
for region in REGIONS[:-1]:
    values = np.sort(static_region_summary.loc[static_region_summary.trajectory_region==region,'min_base_score'])
    ax.plot(values,np.arange(1,len(values)+1)/len(values),label=region,color=REGION_COLORS[region])
ax.set_xscale('symlog',linthresh=.01)
ax.axvline(1,color='black',ls='--',lw=1)
ax.set(xlabel='Minimum distance from static to region (base tolerances)',
       ylabel='Fraction of trajectories',title='Static versus early / middle / late')
ax.legend()
save_show(fig,'02_static_region_distances.png')

PDF: 40/240 trajectories


PDF: 80/240 trajectories


PDF: 120/240 trajectories


PDF: 160/240 trajectories


PDF: 200/240 trajectories


PDF: 240/240 trajectories


## Export and invariants

Preserve input order, metadata, atomic structures, and all DFT labels. Copy the selected EXTXYZ
records without reserializing them. Every removed record must be a hexagonal relaxation frame.
Check all non-target frames, all static frames, formula and trajectory coverage, and quotas.
Write dropped hexagonal frames and per-frame decisions separately. Reopen the outputs and verify
labels, metadata, IDs, hashes, and exact record bytes. Existing results are protected against overwrite.

In [6]:
def read_extxyz_records(path):
    records = []
    with Path(path).open('rb') as handle:
        while True:
            line = handle.readline()
            if not line:
                break
            n_atoms = int(line)
            tail = [handle.readline() for _ in range(n_atoms + 1)]
            assert all(tail), 'Truncated EXTXYZ record'
            records.append(line + b''.join(tail))
    return records


def write_extxyz_records(path, records, indices):
    with Path(path).open('wb') as handle:
        for index in indices:
            handle.write(records[int(index)])

source_records = read_extxyz_records(SOURCE_EXTXYZ)
assert len(source_records) == len(atoms_all)
assert decisions.loc[decisions.polymorph.ne('hexagonal'), 'keep'].all()
assert decisions.loc[~decisions.keep, 'polymorph'].eq('hexagonal').all()
assert decisions.loc[~decisions.keep, 'data_origin'].eq('relax_trajectory').all()
assert decisions.loc[decisions.data_origin.eq('static'),'keep'].all()
assert set(decisions.loc[decisions.keep,'formula']) == set(inventory.formula)
assert set(decisions.loc[decisions.keep,'trajectory_key']) == set(trajectories)
assert stratum_summary.frames_kept.ge(stratum_summary.required_minimum).all()
assert sha256_file(SOURCE_EXTXYZ) == source_hash
kept_atoms, dropped_atoms = [], []
for row in decisions.itertuples(index=False):
    atoms = atoms_all[row.source_index]
    (kept_atoms if row.keep else dropped_atoms).append(atoms)
kept_config_ids = {a.info['config_id'] for a in kept_atoms}
dropped_config_ids = {a.info['config_id'] for a in dropped_atoms}
assert kept_config_ids.isdisjoint(dropped_config_ids)
assert kept_config_ids | dropped_config_ids == set(inventory.config_id)

coverage_rows = []
for (polymorph,region), group in decisions.groupby(['polymorph','trajectory_region']):
    retained = group[group.keep]
    coverage_rows.append(dict(polymorph=polymorph,trajectory_region=region,
        energy_min_before=group.energy_eV_atom.min(), energy_min_after=retained.energy_eV_atom.min(),
        energy_max_before=group.energy_eV_atom.max(), energy_max_after=retained.energy_eV_atom.max(),
        fmax_before=group.fmax_eV_A.max(), fmax_after=retained.fmax_eV_A.max(),
        high_force_p95_retention=group.loc[group.fmax_eV_A.ge(group.fmax_eV_A.quantile(.95)),'keep'].mean()))
coverage_audit = pd.DataFrame(coverage_rows)
display(coverage_audit)

THINNED_EXTXYZ = OUTPUT_DIR/'configurations_only_hex_new.extxyz'
DROPPED_EXTXYZ = OUTPUT_DIR/'dropped_hex_configurations.extxyz'
assert SOURCE_EXTXYZ not in {THINNED_EXTXYZ.resolve(),DROPPED_EXTXYZ.resolve()}
tables = {
    'frame_decisions.csv': decisions,
    'configuration_index.csv': decisions.loc[decisions.keep].reset_index(drop=True),
    'stratum_summary.csv': stratum_summary,
    'trajectory_thinning_summary.csv': trajectory_summary,
    'static_frame_distances.csv': static_frame_distances,
    'static_region_summary.csv': static_region_summary,
    'dataset_composition.csv': composition,
    'coverage_audit.csv': coverage_audit,
}
manifest = {
    'source': str(SOURCE_EXTXYZ), 'source_sha256': source_hash,
    'counts': dict(before=len(atoms_all),after=len(kept_atoms),dropped=len(dropped_atoms),
                   static_retained=len(statics),trajectories=len(trajectories),
                   formulas=int(inventory.formula.nunique())),
    'policy': 'hexagonal_only_geometric_greedy_within_region_then_farthest_point_minimum_quota',
    'target_polymorph': 'hexagonal',
    'non_target_policy': 'keep_all_endpoint_cleaned_frames',
    'records_copied_byte_for_byte': True,
    'historical_non_hex_policies_applied': False,
    'progress': 'ordinal rank among cleaned relax frames / max(n-1,1)',
    'region_boundaries': dict(early_end=EARLY_END,middle_end=MIDDLE_END),
    'base_tolerances': dict(rms_A=BASE_RMS_A,cell_relative=BASE_CELL_REL),
    'quotas': [dict(hexagonal=h,late=l,**p) for (h,l),p in POLICIES.items()],
    'geometry': 'atom-wise MIC in midpoint cell; symmetric Frobenius cell distance',
    'static_always_kept': True, 'split_applied': False, 'labels_used_for_selection': False,
    'randomness': False, 'plot_example': example_key,
    'versions': dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,ase=ase.__version__),
    'output_extxyz': str(THINNED_EXTXYZ), 'disk_verified': False,
}
if WRITE_OUTPUTS:
    write_extxyz_records(THINNED_EXTXYZ, source_records, decisions.loc[decisions.keep, 'source_index'])
    if dropped_atoms:
        write_extxyz_records(DROPPED_EXTXYZ, source_records, decisions.loc[~decisions.keep, 'source_index'])
    else:
        DROPPED_EXTXYZ.write_text('')
    for filename,table in tables.items():
        table.to_csv(REPORT_DIR/filename,index=False)
    (OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
display(pd.Series(manifest['counts']))

,polymorph,trajectory_region,energy_min_before,energy_min_after,energy_max_before,energy_max_after,fmax_before,fmax_after,high_force_p95_retention
0,dist_perovskite,early,-6.682132,-6.682132,-3.668042,-3.668042,1.683140,1.683140,1.000000
1,dist_perovskite,late,-6.683922,-6.683922,-3.673665,-3.673665,1.027938,1.027938,1.000000
2,dist_perovskite,middle,-6.683797,-6.683797,-3.669642,-3.669642,4.851286,4.851286,1.000000
3,dist_perovskite,static,-6.683996,-6.683996,-3.674027,-3.674027,0.056316,0.056316,1.000000
4,hexagonal,early,-6.655288,-6.655288,-2.404266,-2.404266,20.630476,20.630476,0.573171
5,hexagonal,late,-6.674805,-6.674805,-3.307477,-3.307477,2.432759,2.432759,0.695122
6,hexagonal,middle,-6.673264,-6.671501,-3.269610,-3.269610,11.749115,11.749115,0.765690
7,hexagonal,static,-6.674908,-6.674908,-3.290787,-3.290787,0.210179,0.210179,1.000000
8,needle,early,-6.673696,-6.673696,-3.441757,-3.441757,2.249741,2.249741,1.000000
9,needle,late,-6.677653,-6.677653,-3.489151,-3.489151,0.556551,0.556551,1.000000


before             13874
after               8898
dropped             4976
static_retained      240
trajectories         240
formulas              80
dtype: int64

In [7]:
if WRITE_OUTPUTS:
    for path, expected in [(THINNED_EXTXYZ,kept_atoms),(DROPPED_EXTXYZ,dropped_atoms)]:
        restored = read(path,index=':',format='extxyz') if expected else []
        assert len(restored) == len(expected)
        for actual,original in zip(restored,expected):
            assert actual.info.keys() == original.info.keys()
            for key in original.info:
                np.testing.assert_equal(actual.info[key],original.info[key])
            np.testing.assert_array_equal(actual.numbers,original.numbers)
            np.testing.assert_array_equal(actual.pbc,original.pbc)
            for received,source in [(actual.positions,original.positions),(actual.cell.array,original.cell.array),
                                    (actual.get_potential_energy(),original.get_potential_energy()),
                                    (actual.get_forces(),original.get_forces()),
                                    (actual.get_stress(voigt=False),original.get_stress(voigt=False))]:
                np.testing.assert_allclose(received,source,atol=1e-8,rtol=1e-10)
    for path, mask in [(THINNED_EXTXYZ, decisions.keep), (DROPPED_EXTXYZ, ~decisions.keep)]:
        expected_records = [source_records[i] for i in decisions.loc[mask, 'source_index']]
        assert read_extxyz_records(path) == expected_records
    for filename, table in tables.items():
        assert len(pd.read_csv(REPORT_DIR/filename)) == len(table)
    assert sha256_file(SOURCE_EXTXYZ) == source_hash
    paths = [THINNED_EXTXYZ,DROPPED_EXTXYZ]+[REPORT_DIR/name for name in tables]
    paths += [FIGURE_DIR/name for name in ['01_trajectory_screening.png','02_static_region_distances.png',
                                          '03_accumulated_novelty_example_with_static.png']]
    if EXPORT_ALL_TRAJECTORIES_PDF:
        paths.append(FIGURE_DIR/'trajectory_screening_all.pdf')
    manifest['output_sha256'] = {str(p.relative_to(OUTPUT_DIR)): sha256_file(p) for p in paths}
    manifest['disk_verified'] = True
    (OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    print('Verification passed: IDs, order, regions, quotas, metadata, and labels preserved.')
    print('Dataset ready for stratification_split:',THINNED_EXTXYZ)

Verification passed: IDs, order, regions, quotas, metadata, and labels preserved.
Dataset ready for stratification_split: /users/0/cmadridl/projects/dmc_cristelle-madrid-lopez/projects/ABS3-fine-tuning/dev/data/cml/data/Preprocess notebooks/outputs/only_hex_new/configurations_only_hex_new.extxyz


## Next step: reuse the OLD formula split

Run `data/stratification_split.ipynb` after this notebook. Its default input is
`outputs/only_hex_new/configurations_only_hex_new.extxyz`; its output is
`outputs/only_hex_new/formula_split`.

The split reads `configurations_results1_no_meta_formula_split_old/reports/formula_assignment.csv`
and validates it against the historical manifest and EXTXYZ files. Seed 42 is preserved as
historical provenance. The assignment is reused directly, without a new candidate search.
The split checks that needle, dist_perovskite, and static IDs and DFT labels match OLD in each partition.
Region annotations are available in `reports/frame_decisions.csv`, keyed by `config_id`.